In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer

In [3]:
df=pd.read_csv('train.csv',usecols=['Age','Fare','SibSp','Parch','Survived'])

In [4]:
df.head()

,Survived,Age,SibSp,Parch,Fare
0,0,22.0,1,0,7.2500
1,1,38.0,1,0,71.2833
2,1,26.0,0,0,7.9250
3,1,35.0,1,0,53.1000
4,0,35.0,0,0,8.0500


In [5]:
df.isnull().sum()

Survived      0
Age         177
SibSp         0
Parch         0
Fare          0
dtype: int64

In [6]:
df.dropna(inplace=True)

In [7]:
df.isnull().sum()

Survived    0
Age         0
SibSp       0
Parch       0
Fare        0
dtype: int64

In [8]:
df.shape

(714, 5)

In [9]:
df['family']=df['SibSp']+df['Parch']

In [10]:
df.head()

,Survived,Age,SibSp,Parch,Fare,family
0,0,22.0,1,0,7.2500,1
1,1,38.0,1,0,71.2833,1
2,1,26.0,0,0,7.9250,0
3,1,35.0,1,0,53.1000,1
4,0,35.0,0,0,8.0500,0


In [12]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [13]:
df.head()

,Survived,Age,Fare,family
0,0,22.0,7.2500,1
1,1,38.0,71.2833,1
2,1,26.0,7.9250,0
3,1,35.0,53.1000,1
4,0,35.0,8.0500,0


In [14]:
x=df.drop(columns=['Survived'])
y=df['Survived']

In [15]:
x.head()

,Age,Fare,family
0,22.0,7.2500,1
1,38.0,71.2833,1
2,26.0,7.9250,0
3,35.0,53.1000,1
4,35.0,8.0500,0


In [16]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [17]:
x_train,x_test,y_train,y_test=train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=0
)

In [18]:
x_train.head()

,Age,Fare,family
387,36.0,13.0000,0
685,25.0,41.5792,3
20,35.0,26.0000,0
331,45.5,28.5000,0
396,31.0,7.8542,0


In [19]:
## Without Binarization
clf=DecisionTreeClassifier()

clf.fit(x_train,y_train)

y_pred=clf.predict(x_test)

accuracy_score(y_test,y_pred)

0.6083916083916084

In [20]:
np.mean(cross_val_score(DecisionTreeClassifier(),x,y,cv=10,scoring='accuracy'))

np.float64(0.6513497652582159)

In [21]:
# Applying Binarization

from sklearn.preprocessing import Binarizer

In [26]:
## Column Transformation
trf=ColumnTransformer([
    ('bin',Binarizer(copy=False),['family'])
],remainder='passthrough')

In [27]:
x_train_trf=trf.fit_transform(x_train)
x_test_trf=trf.transform(x_test)

In [28]:
x_train_trf

array([[  0.    ,  36.    ,  13.    ],
       [  1.    ,  25.    ,  41.5792],
       [  0.    ,  35.    ,  26.    ],
       ...,
       [  0.    ,  46.    ,  79.2   ],
       [  1.    ,  26.    ,   7.8542],
       [  1.    ,  45.    , 164.8667]], shape=(571, 3))

In [29]:
pd.DataFrame(x_train_trf,columns=['family','Age','Fare'])

,family,Age,Fare
0,0.0,36.0,13.0000
1,1.0,25.0,41.5792
2,0.0,35.0,26.0000
3,0.0,45.5,28.5000
4,0.0,31.0,7.8542
...,...,...,...
566,0.0,28.0,10.5000
567,0.0,19.0,10.5000
568,0.0,46.0,79.2000
569,1.0,26.0,7.8542


In [31]:
clf=DecisionTreeClassifier()

clf.fit(x_train_trf,y_train)
y_pred2=clf.predict(x_test_trf)

accuracy_score(y_test,y_pred2)

0.6293706293706294

In [32]:
x_trf=trf.fit_transform(x)
np.mean(cross_val_score(DecisionTreeClassifier(),x,y,cv=10,scoring='accuracy'))

np.float64(0.651310641627543)